In [ ]:
"""
Exemple d'utilisation du pipeline LightGBM Forecast
Ce script peut être converti en notebook Jupyter
"""

# %% [markdown]
# # 🚀 Pipeline LightGBM Forecast - Exemple d'utilisation
# 
# Ce notebook montre comment utiliser le pipeline pour :
# 1. Entraîner un modèle
# 2. Faire des prédictions
# 3. Analyser les résultats

# %% [markdown]
# ## 📦 Imports

# %%
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.utils import load_config, setup_logging
from src.preprocessing import DataPreprocessor
from src.feature_engineering import FeatureEngineer
from src.model import ModelTrainer
from src.inference import ModelPredictor

# Configuration style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Setup logging
setup_logging(level="INFO")

# %% [markdown]
# ## ⚙️ Configuration

# %%
# Charger config
config = load_config("../config/config.yaml")

print("Configuration chargée:")
print(f"  • Données: {config['data']['raw_path']}")
print(f"  • Split: {config['split']['test_date']}")
print(f"  • Modèle: {config['model']['params']['n_estimators']} arbres")

# %% [markdown]
# ## 1️ ENTRAÎNEMENT
# 
# ### 1.1 Preprocessing

# %%
print("=" * 80)
print("PREPROCESSING")
print("=" * 80)

preprocessor = DataPreprocessor(
    reduction_rate=config['data']['reduction_rate']
)

# Charger données
df = preprocessor.load_data(config['data']['raw_path'])

print(f"\n Aperçu des données:")
print(df.head())
print(f"\nShape: {df.shape}")
print(f"Mémoire: {df.memory_usage(deep=True).sum() / 1e9:.2f} GB")

# %%
# Réduction mémoire
df = preprocessor.reduce_memory(df)

# Créer cible
df = preprocessor.create_target(df, horizon=28)

# Features de base
df = preprocessor.create_base_features(df)

# Segments
df = preprocessor.create_segments(df)

# Préparer
df = preprocessor.prepare_for_training(df)

print(f"\n✅ Preprocessing terminé")
print(f"Shape après preprocessing: {df.shape}")

# %% [markdown]
# ### 1.2 Feature Engineering

# %%
print("\n" + "=" * 80)
print("FEATURE ENGINEERING")
print("=" * 80)

feature_engineer = FeatureEngineer(
    calendar_events=config['features']['calendar_events'],
    lag_windows=config['features']['lag_windows'],
    rolling_windows=config['features']['rolling_windows']
)

df = feature_engineer.create_all_features(df)

print(f"\n✅ Feature engineering terminé")
print(f"Nombre de colonnes: {len(df.columns)}")

# Afficher quelques features créées
print(f"\n📋 Exemples de features créées:")
feature_types = {
    'Lag': [col for col in df.columns if 'lag' in col][:3],
    'Rolling': [col for col in df.columns if 'ma_' in col][:3],
    'Trend': [col for col in df.columns if 'trend_' in col][:3],
    'Événements': [col for col in df.columns if 'solde' in col][:3],
}

for category, features in feature_types.items():
    print(f"\n{category}:")
    for feat in features:
        print(f"  • {feat}")

# %% [markdown]
# ### 1.3 Visualisation des données

# %%
# Distribution de la cible
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df['vues_future_28d_log'].hist(bins=50, ax=axes[0], edgecolor='black')
axes[0].set_title('Distribution de la cible (log)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('vues_future_28d_log')
axes[0].set_ylabel('Fréquence')
axes[0].grid(True, alpha=0.3)

df['segment_popularite'].value_counts().plot(kind='bar', ax=axes[1], 
                                              color=['#2ecc71', '#f39c12', '#e74c3c'])
axes[1].set_title('Distribution des segments', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Segment')
axes[1].set_ylabel('Nombre de recherches')
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

# %% [markdown]
# ### 1.4 Entraînement du modèle

# %%
print("\n" + "=" * 80)
print("ENTRAÎNEMENT")
print("=" * 80)

trainer = ModelTrainer(
    model_params=config['model']['params'],
    early_stopping_rounds=config['model']['early_stopping_rounds'],
    feature_selection_threshold=config['model']['feature_selection_threshold']
)

# Split train/test
X_train, X_test, y_train, y_test = trainer.prepare_data(
    df, 
    test_date=config['split']['test_date']
)

# Target encoding
X_train, X_test = trainer.apply_target_encoding(
    X_train, X_test, y_train,
    smoothing=config['encoding']['smoothing'],
    min_samples=config['encoding']['min_samples']
)

# Normalisation
X_train, X_test = trainer.apply_normalization(X_train, X_test)

# Sélection features
selected_features = trainer.select_features(
    X_train, y_train,
    validation_size=config['split']['validation_size']
)

# Entraînement
trainer.train(
    X_train, y_train,
    validation_size=config['split']['validation_size']
)

print(f"\n✅ Entraînement terminé")

# %% [markdown]
# ### 1.5 Évaluation

# %%
metrics = trainer.evaluate(X_test, y_test)

print(f"\n📊 MÉTRIQUES:")
for metric, value in metrics.items():
    print(f"  • {metric.upper()}: {value:.4f}")

# %%
# Visualisation feature importance
feature_importance = pd.DataFrame({
    'feature': selected_features,
    'importance': trainer.model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(12, 8))
feature_importance.head(20).plot(x='feature', y='importance', kind='barh', 
                                  color='#3498db', edgecolor='black')
plt.title('Top 20 Features les plus importantes', fontsize=14, fontweight='bold')
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.gca().invert_yaxis()
plt.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

# %% [markdown]
# ### 1.6 Sauvegarde

# %%
# Sauvegarder modèle
trainer.save(
    model_path=config['output']['model_path'],
    artifacts_path=config['output']['artifacts_path']
)

print(f"✅ Modèle sauvegardé")

# %% [markdown]
# ## 2️⃣ PRÉDICTION

# %%
print("\n" + "=" * 80)
print("PRÉDICTION")
print("=" * 80)

# Charger prédicteur
predictor = ModelPredictor(
    model_path=config['output']['model_path'],
    artifacts_path=config['output']['artifacts_path'],
    config=config
)

# Charger nouvelles données (ici on utilise X_test pour l'exemple)
# Dans la pratique, charger de vraies nouvelles données
new_data = df[df.index >= config['split']['test_date']].copy()

print(f"Prédiction sur {len(new_data):,} observations...")

# Prédictions
predictions = predictor.predict(new_data)

print(f"\n✅ Prédictions générées")
print(f"\n📊 Aperçu:")
print(predictions.head(10))

# %% [markdown]
# ### 2.1 Top K Recherches

# %%
# Top 30 recherches
top_30 = predictor.predict_top_k(new_data, k=30)

print(f"\n🏆 TOP 30 RECHERCHES:")
print(top_30)

# Visualisation
plt.figure(figsize=(12, 8))
plt.barh(range(30), top_30['vues_predites'].values, color='#2ecc71', edgecolor='black')
plt.yticks(range(30), top_30['search_id'].values, fontsize=8)
plt.xlabel('Vues Prédites', fontsize=12, fontweight='bold')
plt.title('Top 30 Recherches - Vues Prédites', fontsize=14, fontweight='bold')
plt.gca().invert_yaxis()
plt.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

# %% [markdown]
# ### 2.2 Analyse des prédictions

# %%
# Statistiques
print(f"\n📈 STATISTIQUES DES PRÉDICTIONS:")
print(f"  • Nombre: {len(predictions):,}")
print(f"  • Moyenne: {predictions['vues_predites'].mean():.1f}")
print(f"  • Médiane: {predictions['vues_predites'].median():.1f}")
print(f"  • Min: {predictions['vues_predites'].min():.1f}")
print(f"  • Max: {predictions['vues_predites'].max():.1f}")

# Distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

predictions['vues_predites'].hist(bins=50, ax=axes[0], edgecolor='black', color='#3498db')
axes[0].set_title('Distribution des vues prédites', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Vues prédites')
axes[0].set_ylabel('Fréquence')
axes[0].grid(True, alpha=0.3)

predictions['vues_predites_log'].hist(bins=50, ax=axes[1], edgecolor='black', color='#e74c3c')
axes[1].set_title('Distribution des vues prédites (log)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Vues prédites (log)')
axes[1].set_ylabel('Fréquence')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# %% [markdown]
# ## 3️⃣ ANALYSE DES RÉSULTATS

# %%
# Comparer prédictions vs réalité (si on a les vraies valeurs)
if 'vues_future_28d' in new_data.columns:
    results_df = pd.DataFrame({
        'date': new_data.index,
        'search_id': new_data['search_id'],
        'vues_reelles': new_data['vues_future_28d'],
        'vues_predites': predictions['vues_predites']
    })
    
    # Calculer erreurs
    results_df['erreur'] = results_df['vues_reelles'] - results_df['vues_predites']
    results_df['erreur_pct'] = (results_df['erreur'] / results_df['vues_reelles']) * 100
    
    print(f"\n ANALYSE DES ERREURS:")
    print(f"  • Erreur moyenne: {results_df['erreur'].mean():.1f}")
    print(f"  • Erreur médiane: {results_df['erreur'].median():.1f}")
    print(f"  • Erreur % moyenne: {results_df['erreur_pct'].abs().mean():.1f}%")
    
    # Visualisation
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Scatter plot
    sample = results_df.sample(min(1000, len(results_df)))
    axes[0].scatter(sample['vues_reelles'], sample['vues_predites'], 
                    alpha=0.5, s=20, color='#3498db')
    max_val = max(sample['vues_reelles'].max(), sample['vues_predites'].max())
    axes[0].plot([0, max_val], [0, max_val], 'r--', linewidth=2, label='Parfait')
    axes[0].set_xlabel('Vues Réelles')
    axes[0].set_ylabel('Vues Prédites')
    axes[0].set_title('Prédictions vs Réalité', fontsize=14, fontweight='bold')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Erreurs
    results_df['erreur'].hist(bins=50, ax=axes[1], edgecolor='black', color='#e74c3c')
    axes[1].axvline(0, color='black', linestyle='--', linewidth=2)
    axes[1].set_xlabel('Erreur (Réel - Prédit)')
    axes[1].set_ylabel('Fréquence')
    axes[1].set_title('Distribution des erreurs', fontsize=14, fontweight='bold')
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

# %% [markdown]
# ## 📝 Conclusion
# 
# Le pipeline a été exécuté avec succès ! 
# 
# **Prochaines étapes possibles :**
# - Ajuster les hyperparamètres dans `config/config.yaml`
# - Ajouter de nouveaux événements calendaires
# - Créer des features custom
# - Analyser les erreurs par segment
# - Utiliser le modèle en production

print("\n" + "=" * 80)
print("✅ PIPELINE TERMINÉ AVEC SUCCÈS")
print("=" * 80)